# Encoder model 

In this section, we will discuss encoder models, with a focus on BERT (Bidirectional Encoder Representations from Transformers) as an example. We will cover the architecture, pre-training, and fine-tuning processes.

BERT is an encoder model that uses a vocabulary of 30,000 tokens. Input tokens are
converted to 1024-dimensional word embeddings and passed through 24 transformer
layers. Each contains a self-attention mechanism with 16 heads. The queries, keys, and
values for each head are of dimension 64 (i.e., the matrices $\Omega_{vh}, \Omega_{qh}, \Omega_{kh}$ are 64 ×1024).

The dimension of the single hidden layer in the fully connected networks is 4096. The
total number of parameters is ∼340 million. When BERT was introduced, this was
considered large, but it is now much smaller than state-of-the-art models.
Encoder models like BERT exploit transfer learning (section 9.3.6). During pre-
training, the parameters of the transformer architecture are learned using self-supervision
from a large corpus of text. The goal here is for the model to learn general information
about the statistics of language. In the fine-tuning stage, the resulting network is adapted
to solve a particular task using a smaller body of labelled training data.

### Pre-training 

In the pre-training stage, the network is trained using self-supervision. This allows the
use of enormous amounts of data without the need for manual labels. For BERT, the self-
supervision task consists of predicting missing words from sentences from a large internet
corpus (figure 12.10).1 During training, the maximum input length is 512 tokens, and
the batch size is 256. The system is trained for a million steps, corresponding to roughly
50 epochs of the 3.3-billion word corpus.
Predicting missing words forces the transformer network to understand some syntax.
For example, it might learn that the adjective red is often found before nouns like house
or car but never before a verb like shout. It also allows the model to learn superficial
common sense about the world. For example, after training, the model will assign a
higher probability to the missing word train in the sentence The <mask> pulled into
the station than it would to the word peanut. However, the degree of “understanding”
this type of model can ever have is limited.


<img src="assets/Chap12/TransformerEncoder.svg" style="filter: invert(1);" width="100%">

Pre-training for BERT-like encoder. The input tokens (and a special
<cls> token denoting the start of the sequence) are converted to word embeddings.
Here, these are represented as rows rather than columns, so the box
labeled “word embeddings” is XT . These embeddings are passed through a series
of transformer layers (orange connections indicate that every token attends
to every other token in these layers) to create a set of output embeddings. A
small fraction of the input tokens are randomly replaced with a generic <mask>
token. In pre-training, the goal is to predict the missing word from the associated
output embedding. To this end, the outputs corresponding to the masked tokens
are passed through softmax functions, and a multiclass classification loss (section
5.24) is applied to each. This task has the advantage that it uses both the
left and right context to predict the missing word but has the disadvantage that
it does not make efficient use of data; here, seven tokens need to be processed to
add two terms to the loss function.

<img src="assets/Chap12/TransformerEncoderFineTune.svg" style="filter: invert(1);" width="100%">

After pre-training, the encoder is fine-tuned using manually labeled
data to solve a particular task. Usually, a linear transformation or a multi-layer
perceptron (MLP) is appended to the encoder to produce whatever output is
required. a) Example text classification task. In this sentiment classification
task, the <cls> token embedding is used to predict the probability that the
review is positive. b) Example word classification task. In this named entity
recognition problem, the embedding for each word is used to predict whether the
word corresponds to a person, place, or organization, or is not an entity.

### Fine-tuning

In the fine-tuning stage, the model parameters are adjusted to specialize the network to
a particular task. An extra layer is appended onto the transformer network to convert
the output vectors to the desired output format. Examples include:

**Text classification**: In BERT, a special token known as the classification or `<cls>`
token is placed at the start of each string during pre-training. For text classification
tasks like <u>sentiment analysis</u> (in which the passage is labeled as having a positive or
negative emotional tone), the vector associated with the `<cls>` token is mapped to a
single number and passed through a logistic sigmoid (figure 12.11a). This contributes to
a standard binary cross-entropy loss (section 5.4).

**Word classification**: The goal of <u>named entity recognition</u> is to classify each word as an entity type (e.g., person, place, organization, or no-entity). To this end, each input
embedding $\mathbf{x}_n$ is mapped to an $E\times 1$ vector where the $E$ entries correspond to the $E$ entity types. This is passed through a softmax function to create probabilities for each class, which contribute to a multiclass cross-entropy loss (figure 12.11b).

**Text span prediction**: In the SQuAD 1.1 question answering task, the question and a
passage from Wikipedia containing the answer are concatenated and tokenized. BERT
is then used to predict the text span in the passage that contains the answer. Each
token maps to two numbers indicating how likely it is that the text span begins and
ends at this location. The resulting two sets of numbers are put through two softmax
functions. The likelihood of any text span being the answer can be derived by combining
the probability of starting and ending at the appropriate places.


In [ ]:
import re, collections
     

text = "a sailor went to sea sea sea "+\
                  "to see what he could see see see "+\
                  "but all that he could see see see "+\
                  "was the bottom of the deep blue sea sea sea"

Tokenize the input sentence To begin with the tokens are the individual letters and the whitespace token. So, we represent each word in terms of these tokens with spaces between the tokens to delineate them.

The tokenized text is stored in a structure that represents each word as tokens together with the count of how often that word occurs. We'll call this the vocabulary.

In [ ]:
def initialize_vocabulary(text):
  vocab = collections.defaultdict(int)
  words = text.strip().split()
  for word in words:
      vocab[' '.join(list(word)) + ' '] += 1
  return vocab
     

vocab = initialize_vocabulary(text)
print('Vocabulary: {}'.format(vocab))
print('Size of vocabulary: {}'.format(len(vocab)))

Find all the tokens in the current vocabulary and their frequencies

In [ ]:
def get_tokens_and_frequencies(vocab):
  tokens = collections.defaultdict(int)
  for word, freq in vocab.items():
      word_tokens = word.split()
      for token in word_tokens:
          tokens[token] += freq
  return tokens
     

tokens = get_tokens_and_frequencies(vocab)
print('Tokens: {}'.format(tokens))
print('Number of tokens: {}'.format(len(tokens)))

Find each pair of adjacent tokens in the vocabulary and count them. We will subsequently merge the most frequently occurring pair.

In [ ]:
def get_pairs_and_counts(vocab):
    pairs = collections.defaultdict(int)
    for word, freq in vocab.items():
        symbols = word.split()
        for i in range(len(symbols)-1):
            pairs[symbols[i],symbols[i+1]] += freq
    return pairs
     

pairs = get_pairs_and_counts(vocab)
print('Pairs: {}'.format(pairs))
print('Number of distinct pairs: {}'.format(len(pairs)))

most_frequent_pair = max(pairs, key=pairs.get)
print('Most frequent pair: {}'.format(most_frequent_pair))

Merge the instances of the best pair in the vocabulary

In [ ]:
def merge_pair_in_vocabulary(pair, vocab_in):
    vocab_out = {}
    bigram = re.escape(' '.join(pair))
    p = re.compile(r'(?

vocab = merge_pair_in_vocabulary(most_frequent_pair, vocab)
print('Vocabulary: {}'.format(vocab))
print('Size of vocabulary: {}'.format(len(vocab)))

Update the tokens, which now include the best token 'se'

In [ ]:
tokens = get_tokens_and_frequencies(vocab)
print('Tokens: {}'.format(tokens))
print('Number of tokens: {}'.format(len(tokens)))

Now let's write the full tokenization routine

In [ ]:
# TODO -- write this routine by filling in this missing parts,
# calling the above routines
def tokenize(text, num_merges):
  # Initialize the vocabulary from the input text
  # vocab = (your code here)

  for i in range(num_merges):
    # Find the tokens and how often they occur in the vocabulary
    # tokens = (your code here)

    # Find the pairs of adjacent tokens and their counts
    # pairs = (your code here)

    # Find the most frequent pair
    # most_frequent_pair = (your code here)
    print('Most frequent pair: {}'.format(most_frequent_pair))

    # Merge the code in the vocabulary
    # vocab = (your code here)

  # Find the tokens and how often they occur in the vocabulary one last time
  # tokens = (your code here)

  return tokens, vocab
     

tokens, vocab = tokenize(text, num_merges=22)
     

print('Tokens: {}'.format(tokens))
print('Number of tokens: {}'.format(len(tokens)))
print('Vocabulary: {}'.format(vocab))
print('Size of vocabulary: {}'.format(len(vocab)))

TODO - Consider the input text:

"How much wood could a woodchuck chuck if a woodchuck could chuck wood"

How many tokens will there be initially and what will they be? How many tokens will there be if we run the tokenization routine for the maximum number of iterations (merges)?

When you've made your predictions, run the code and see if you are correct.